In [8]:
pip install -qU langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
pip install langchain   tiktoken pypdf  langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
pip install -qU faiss-cpu langchain-community langchain-core

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_core.documents import Document

# 5 create our list of 5 play documents

# Create our list of 5 player documents with rich metadata
player_docs = [
    Document(
        page_content="Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.",
        metadata={"name": "Lionel Messi", "sport": "Football/Soccer", "active": True}
    ),
    Document(
        page_content="LeBron James is an American professional basketball player who has won multiple NBA championships and MVP awards.",
        metadata={"name": "LeBron James", "sport": "Basketball", "active": True}
    ),
    Document(
        page_content="Serena Williams is an American former professional tennis player who won 23 Grand Slam women's singles titles.",
        metadata={"name": "Serena Williams", "sport": "Tennis", "active": False}
    ),
    Document(
        page_content="Babar Azam is a Pakistani international cricketer and a top-order batter, known for his elegant cover drives.",
        metadata={"name": "Babar Azam", "sport": "Cricket", "active": True}
    ),
    Document(
        page_content="Muhammad Ali was an American professional boxer and activist, nicknamed 'The Greatest'.",
        metadata={"name": "Muhammad Ali", "sport": "Boxing", "active": False}
    )
]

print(f"Successfully created {len(player_docs)} documents")

Successfully created 5 documents


In [5]:
# generate embaddings
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Initialize our embedding model (same as before)
embeddings_of_doc = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'}, # Change to 'cuda' if using Colab GPU
    encode_kwargs={'normalize_embeddings': True}
)





c:\Users\Administrator\Desktop\GenAI - Langchain\Gen AI\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3955.07it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# 2 . Create Faiss vector database
from langchain_community.vectorstores import FAISS
vectore_store = FAISS.from_documents(player_docs,embeddings_of_doc)
vectore_store.save_local("Faiss_db")
print("Database is successfully build")

Database is successfully build


In [12]:
# add document

vectore_store.add_documents(player_docs)

['c66ea79a-e598-400c-8b27-1682e7994e36',
 'ef875254-cfc8-4bb2-8fe9-c2c2b1c2f540',
 '74f7bee9-e257-4b83-bb97-370af8bce324',
 '68e5f600-fd54-4832-96a1-6313fadd450d',
 'f17da8d3-b070-4e06-9b5d-ed8b7c6cfb70']

In [13]:
from langchain_community.vectorstores import FAISS

# 1. Load the saved database from your folder
# You must pass the same embedding model you used to create it
# allow_dangerous_deserialization=True is required by LangChain because FAISS uses .pkl files.
# It is completely safe as long as you created the database yourself!
loaded_vector_store = FAISS.load_local(
    folder_path="Faiss_db",
    embeddings=embeddings_of_doc,
    allow_dangerous_deserialization=True
)

# 2. Access the documents stored inside the database's internal dictionary
stored_docs = loaded_vector_store.docstore._dict.values()

print(f"Successfully loaded {len(stored_docs)} documents from the database!\n")

# 3. Loop through and print each document's content and metadata
for i, doc in enumerate(stored_docs, 1):
    print(f"--- Document {i} ---")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")

Successfully loaded 5 documents from the database!

--- Document 1 ---
Content: Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.
Metadata: {'name': 'Lionel Messi', 'sport': 'Football/Soccer', 'active': True}

--- Document 2 ---
Content: LeBron James is an American professional basketball player who has won multiple NBA championships and MVP awards.
Metadata: {'name': 'LeBron James', 'sport': 'Basketball', 'active': True}

--- Document 3 ---
Content: Serena Williams is an American former professional tennis player who won 23 Grand Slam women's singles titles.
Metadata: {'name': 'Serena Williams', 'sport': 'Tennis', 'active': False}

--- Document 4 ---
Content: Babar Azam is a Pakistani international cricketer and a top-order batter, known for his elegant cover drives.
Metadata: {'name': 'Babar Azam', 'sport': 'Cricket', 'active': True}

--- Document 5 ---
Content: Muhammad Ali was an American profes

In [14]:
# 1. Access the underlying raw FAISS index object
faiss_index = loaded_vector_store.index

# 2. Check how many embeddings (vectors) are stored
total_vectors = faiss_index.ntotal
print(f"Total embeddings stored in the index: {total_vectors}\n")

# 3. Loop through and view the actual numerical arrays
for i in range(total_vectors):
    # reconstruct(i) pulls the raw vector array out of the FAISS memory
    vector = faiss_index.reconstruct(i)

    # Let's also grab the associated document so we know WHO this math belongs to
    doc_id = loaded_vector_store.index_to_docstore_id[i]
    document = loaded_vector_store.docstore.search(doc_id)

    print(f"--- Embedding {i+1} ---")
    print(f"Belongs to Document: {document.metadata.get('name', 'Unknown')}")
    print(f"Total Dimensions: {len(vector)}")

    # We only print the first 5 numbers. Printing all 384+ numbers per document
    # would flood your Colab output screen!
    print(f"Vector Data (first 5 numbers): {vector[:5]}\n")

Total embeddings stored in the index: 5

--- Embedding 1 ---
Belongs to Document: Lionel Messi
Total Dimensions: 384
Vector Data (first 5 numbers): [ 0.01487996  0.07624785  0.04286608 -0.0453329   0.04222475]

--- Embedding 2 ---
Belongs to Document: LeBron James
Total Dimensions: 384
Vector Data (first 5 numbers): [-0.05190973  0.00701101  0.02326682 -0.0820135   0.00126943]

--- Embedding 3 ---
Belongs to Document: Serena Williams
Total Dimensions: 384
Vector Data (first 5 numbers): [ 0.01500528  0.00088995 -0.02488973  0.02632111  0.02880733]

--- Embedding 4 ---
Belongs to Document: Babar Azam
Total Dimensions: 384
Vector Data (first 5 numbers): [-0.01752556  0.04191956  0.03683588 -0.0165595   0.00398721]

--- Embedding 5 ---
Belongs to Document: Muhammad Ali
Total Dimensions: 384
Vector Data (first 5 numbers): [-0.04445593  0.07632949 -0.01653677 -0.02423165 -0.03690405]



In [15]:
# search docments
vectore_store.similarity_search(
    query = " who is linol Messi?",
    k=1
)

[Document(id='faf68dee-0e69-4b5e-892b-bc4643acb9fc', metadata={'name': 'Lionel Messi', 'sport': 'Football/Soccer', 'active': True}, page_content='Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.')]

In [16]:
vectore_store.similarity_search_with_score(
    query = " who is linol Messi?",
    k=1
)

[(Document(id='faf68dee-0e69-4b5e-892b-bc4643acb9fc', metadata={'name': 'Lionel Messi', 'sport': 'Football/Soccer', 'active': True}, page_content='Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.'),
  np.float32(0.3738883))]

In [17]:
vectore_store.similarity_search_with_score(
    query = "",
    filer={'sport': 'Football/Soccer'}
)

[(Document(id='ba19cef3-68ac-482e-90ea-595c2ccd3486', metadata={'name': 'LeBron James', 'sport': 'Basketball', 'active': True}, page_content='LeBron James is an American professional basketball player who has won multiple NBA championships and MVP awards.'),
  np.float32(0.9612331)),
 (Document(id='ef875254-cfc8-4bb2-8fe9-c2c2b1c2f540', metadata={'name': 'LeBron James', 'sport': 'Basketball', 'active': True}, page_content='LeBron James is an American professional basketball player who has won multiple NBA championships and MVP awards.'),
  np.float32(0.9612331)),
 (Document(id='faf68dee-0e69-4b5e-892b-bc4643acb9fc', metadata={'name': 'Lionel Messi', 'sport': 'Football/Soccer', 'active': True}, page_content='Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.'),
  np.float32(1.0724474)),
 (Document(id='c66ea79a-e598-400c-8b27-1682e7994e36', metadata={'name': 'Lionel Messi', 'sport': 'Football/Soccer', 